# Decision trees

A decision tree is a flowchart of yes/no questions learned from data. It is the one model here that a non-technical colleague can read, and it is the building block of random forests and gradient boosting in the next notebook. It also overfits very easily, which is the lesson.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

from common.datasets import load_breast_cancer_data, load_iris_flowers
from common.plotting import plot_decision_boundary

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

Twenty questions. To identify a flower you ask "is the petal shorter than 2.5 cm?" and branch on the answer, then ask another question inside each branch, until the group left is nearly all one species. Each question splits one feature at one threshold, so the boundary is a set of axis-aligned boxes.

Two things need deciding at every step: which question to ask, and when to stop. A good question leaves the two groups **purer** than before. Stopping late gives a tree that memorises, stopping early gives one that is too crude.

Trees need no scaling, handle mixed units, capture interactions (a split on $B$ inside the branch for $A$) and are easy to read. They pay for it with instability: a small change in the data can change the whole tree.

## 2. Maths

**Impurity.** For a node where the class shares are $p_1,\dots,p_K$:

$$\text{Gini}=1-\sum_k p_k^2\qquad\text{Entropy}=-\sum_k p_k\log_2 p_k$$

Both are 0 for a pure node and largest when classes are evenly mixed (for two classes, 0.5 and 1 bit). Gini is the chance of mislabelling a random point if you label it by drawing from the node's class shares. Entropy is the average number of bits needed to say which class a point belongs to (see `02_probability`). They almost always choose the same splits.

**Cost function.** For a split of a node with $n$ rows into a left part with $n_L$ and a right part with $n_R$, the cost is the weighted impurity of the children:

$$J(\text{feature},t)=\frac{n_L}{n}G_L+\frac{n_R}{n}G_R$$

The tree picks the feature and threshold with the lowest $J$, which is the largest **impurity decrease** $G-J$. It is a greedy search: it takes the best split now, never looking ahead, then recurses on each child. Finding the best tree overall is NP-hard, so greedy is the standard compromise.

**Candidate thresholds.** For one feature, sort the values and try the midpoint between each pair of neighbours. That is at most $n-1$ candidates per feature.

**Regularisation.** Left alone, a tree keeps splitting until every leaf is pure and training accuracy is 100%. We limit it with `max_depth`, `min_samples_leaf`, `min_samples_split`, or prune afterwards with **cost-complexity pruning**, which minimises $\text{error}+\alpha\cdot(\text{number of leaves})$. That is the same penalty idea as notebook 13, with leaf count as the complexity.

## 3. From scratch

Gini and entropy first, then the exhaustive search for the best split on a feature.

In [ ]:
def gini(y: np.ndarray) -> float:
    p = np.bincount(y) / len(y)
    return float(1 - np.sum(p**2))


def entropy(y: np.ndarray) -> float:
    p = np.bincount(y) / len(y)
    p = p[p > 0]  # 0 * log 0 is taken as 0
    return float(-np.sum(p * np.log2(p))) + 0.0  # + 0.0 turns -0.0 into 0.0


for counts in ([10, 0], [9, 1], [7, 3], [5, 5]):
    y = np.repeat([0, 1], counts)
    print(f"{counts}: gini {gini(y):.3f}  entropy {entropy(y):.3f}")

Both rise from 0 at a pure node to the maximum at 50/50. Now the split search. For each candidate threshold, compute the weighted child impurity and keep the lowest.

In [ ]:
def best_split(x: np.ndarray, y: np.ndarray, impurity=gini) -> tuple[float, float]:
    """Return (threshold, weighted child impurity) for the best split of one feature."""
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    best_t, best_cost = np.nan, np.inf
    for i in range(1, len(xs)):
        if xs[i] == xs[i - 1]:
            continue  # no threshold between equal values
        cost = (i * impurity(ys[:i]) + (len(ys) - i) * impurity(ys[i:])) / len(ys)
        if cost < best_cost:
            best_t, best_cost = (xs[i] + xs[i - 1]) / 2, cost
    return float(best_t), float(best_cost)


X_iris, y_iris = load_iris_flowers()
yi = y_iris.to_numpy()
print("parent gini:", round(gini(yi), 3))
for col in X_iris.columns:
    t, cost = best_split(X_iris[col].to_numpy(), yi)
    print(f"{col:>13}: threshold {t:5.2f}  child gini {cost:.3f}  decrease {gini(yi) - cost:.3f}")

Petal length (or petal width, which ties) wins. The decrease from 0.667 down to 0.333 comes from isolating setosa completely with one cut. Check that sklearn's root split agrees.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(X_iris, y_iris)
print(export_text(tree, feature_names=list(X_iris.columns)))

The root is petal length at 2.45 cm, exactly what the search found (petal width at 0.8 cm ties), and it puts all 50 setosa on one side. The second split separates versicolor from virginica. Each leaf shows its class counts, which are the class probabilities.

I leave the recursion to sklearn. It is the same loop applied to each child with the rows that reached it, plus the stopping rules below.

## 4. The library version and the boundary

With two features we can draw the boxes. Compare a depth-2 tree with an unrestricted one.

In [ ]:
cols = ["petal_length", "petal_width"]
X2 = X_iris[cols].to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, depth in zip(axes, (2, 4, None)):
    m = DecisionTreeClassifier(max_depth=depth, random_state=SEED).fit(X2, yi)
    plot_decision_boundary(m, X2, yi, ax, labels=("petal length", "petal width"))
    ax.set_title(f"max_depth={depth}: {m.get_n_leaves()} leaves")
plt.show()

Every boundary is a horizontal or vertical line, and the deeper tree carves small boxes around the odd points where versicolor and virginica overlap. Those boxes fit the training data and say nothing reliable about new flowers.

## 5. Depth and overfitting

Breast cancer, 30 features. Train and cross-validated accuracy against depth shows where the tree stops learning and starts memorising.

In [ ]:
Xb, yb = load_breast_cancer_data()
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=SEED, stratify=yb)
rows = []
for depth in range(1, 13):
    m = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    rows.append(
        {
            "depth": depth,
            "train": m.fit(Xb_tr, yb_tr).score(Xb_tr, yb_tr),
            "cv": cross_val_score(m, Xb_tr, yb_tr, cv=5).mean(),
        }
    )
depth_table = pd.DataFrame(rows).set_index("depth")
depth_table.plot(marker="o")
plt.ylabel("accuracy")
plt.show()
depth_table.round(3).T

Training accuracy climbs to 1.0 and stays there. Cross-validated accuracy is flat at about 0.92 to 0.93 from depth 2 onwards and never improves, so every extra level is memorisation. The widening gap between the two lines is the overfitting. With 398 training rows the noise in these CV numbers is about a point, so do not read much into the exact peak.

**Other controls.** `min_samples_leaf` refuses splits that leave a leaf with too few rows, so no leaf rests on 1 or 2 points. `ccp_alpha` prunes after growing. Try both against the unrestricted tree on the held-out test set.

In [ ]:
configs = {
    "unrestricted": {},
    "max_depth=3": {"max_depth": 3},
    "min_samples_leaf=10": {"min_samples_leaf": 10},
    "ccp_alpha=0.01": {"ccp_alpha": 0.01},
}
rows = []
for name, kw in configs.items():
    m = DecisionTreeClassifier(random_state=SEED, **kw).fit(Xb_tr, yb_tr)
    rows.append(
        {
            "config": name,
            "leaves": m.get_n_leaves(),
            "train": m.score(Xb_tr, yb_tr),
            "test": m.score(Xb_te, yb_te),
        }
    )
pd.DataFrame(rows).set_index("config").round(3)

The unrestricted tree has 19 leaves, a perfect training score and 0.901 on test. The regularised trees have 5 to 8 leaves, a lower training score, and 0.92 to 0.94 on test. Pick the control by cross-validation, not by looking at the test set. For `ccp_alpha`, `tree.cost_complexity_pruning_path(X, y)` lists the useful values to try.

## 6. Feature importance

A tree can say which features it used. The importance of a feature is the total impurity decrease from every split on it, weighted by the rows reaching each split and normalised to sum to 1.

In [ ]:
m = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xb_tr, yb_tr)
imp = pd.Series(m.feature_importances_, index=Xb.columns).sort_values(ascending=False)
print(imp.head(6).round(3))
fig, ax = plt.subplots(figsize=(11, 4.5))
plot_tree(
    m,
    feature_names=list(Xb.columns),
    class_names=["benign", "malignant"],
    filled=True,
    impurity=False,
    fontsize=7,
    ax=ax,
)
plt.show()

A handful of features carry nearly all the importance, and the tree diagram can be read top to bottom as a rule set. Two cautions. The importance is computed on training data, so a noise feature can score above zero in an overfitted tree. And when two features are correlated (as `radius`, `perimeter` and `area` are here), the tree picks one and the other looks unimportant. Permutation importance on held-out data is the safer check.

## 7. What breaks: instability

Because each split is chosen greedily from the exact data, small changes in the sample can flip the top split and change every branch below it. Resample the training rows 8 times and look at the root question each time.

In [ ]:
rows = []
for s in range(8):
    idx = np.random.default_rng(s).choice(len(Xb_tr), size=len(Xb_tr), replace=True)
    t = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xb_tr.iloc[idx], yb_tr.iloc[idx])
    root = t.tree_.feature[0]
    rows.append(
        {
            "resample": s,
            "root feature": Xb.columns[root],
            "threshold": round(t.tree_.threshold[0], 2),
            "test acc": round(t.score(Xb_te, yb_te), 3),
        }
    )
pd.DataFrame(rows).set_index("resample")

The root is `worst perimeter` in five of the eight, `worst radius` in two (a near-copy of the same information) and `mean concave points` in one, with the threshold moving each time. Test accuracy ranges from 0.889 to 0.942. A single tree is a high-variance model. The fix is not a smarter tree. It is to train many trees on different resamples and average them, which is the next notebook.

**Other limits.** Trees cannot extrapolate beyond the range of the training data (a regression tree predicts a constant per leaf). Axis-aligned boxes approximate a diagonal boundary with a staircase, which costs many splits. A tree with a very imbalanced target needs `class_weight` or a different metric (notebook 25).